# LoRA Fine-Tuning: Model Adaptation with Parameter-Efficient Adaptation



In this notebook, we will walk through a **LoRA (Low-Rank Adaptation) fine-tuning pipeline** on a small language model. By the end, you will have:

1. **Understood** what LoRA does and why it is preferred over full fine-tuning.
2. **Applied** LoRA adapters to a pre-trained T5 model and inspected the parameter savings.
3. **Prepared** a small dataset for supervised fine-tuning.
4. **Trained** the LoRA-adapted model with a standard training loop.
5. **Compared** the outputs of the base model vs. the LoRA-tuned model at inference time.
6. **Saved and loaded** LoRA adapter weights independently of the base model.


Let's install the required libraries. We will use:

- **`transformers`**: Hugging Face's library for loading pre-trained models and tokenizers.
- **`peft`**: Hugging Face's Parameter-Efficient Fine-Tuning library, which provides the LoRA implementation.
- **`datasets`**: for loading and preparing training data.
- **`accelerate`**: for efficient device placement and mixed-precision support.

These are the standard tools used in industry for LoRA-based fine-tuning.

In [1]:
!pip install -q transformers peft datasets accelerate

### LoRA Recap: Why Not Update All Parameters?

Before diving into code, it is important for us to recall why LoRA exists.

### The problem with full fine-tuning

A model like T5-Small has **~60 million parameters**. Larger models (7B, 70B, 405B) have billions. Full fine-tuning updates **every single parameter**, which means:

- **High GPU memory:** gradients and optimizer states for all parameters must fit in memory.
- **Catastrophic forgetting:** the model can overwrite its general knowledge with task-specific patterns.
- **One copy per task:** each fine-tuned variant is a full copy of the model.

### What LoRA does instead

LoRA **freezes** all original model weights and injects small, trainable **low-rank matrices** into selected layers (typically the attention projections). For a weight matrix **W** of size `d × k`:

```
W_new = W (frozen) + B · A
```

Where:
- **A** has shape `d × r` and **B** has shape `r × k`, with **r** (the rank) being very small (e.g. 8 or 16).
- Only **A** and **B** are trained; this is a tiny fraction of total parameters.
- At inference, the adapter can be **merged** into the base weights (zero overhead) or kept separate (swappable).

Learners will see this parameter difference concretely in the cells that follow.

### Load the Base Model and Tokenizer

We will use **T5-Small**, a 60M-parameter encoder-decoder model from Google. T5 treats every NLP task as a text-to-text problem: the input is a text string and the output is a text string. This makes it ideal for demonstrating fine-tuning; we can see exactly what the model generates before and after adaptation.

The cell below loads the model and its tokenizer. The model is moved to GPU if available.

In [2]:
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer

MODEL_NAME = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
base_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model = base_model.to(device)

print(f"Model loaded: {MODEL_NAME}")
print(f"Device: {device}")
print(f"Total parameters: {sum(p.numel() for p in base_model.parameters()):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: t5-small
Device: cuda
Total parameters: 60,506,624


### Baseline Inference (Before Fine-Tuning)

Before applying LoRA, we should observe how the **base T5-Small** model responds to the kind of task we will fine-tune on. This gives an idea of the extent of fine-tuning needed and establishes a **baseline** so the effect of LoRA training is clearly visible later.

The task chosen for this demo is **grammatical error correction (GEC)**. The model receives a sentence with grammatical errors and should output the corrected version. T5-Small has not been specifically trained for this, so we should expect mediocre or incorrect outputs at this stage.

In [3]:
def generate(model, text, max_length=128):
    inputs = tokenizer(text, return_tensors="pt", max_length=256, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_sentences = [
    "fix grammar: She go to the store yesterday and buyed milk.",
    "fix grammar: Their is many reason why peoples should exercize daily.",
    "fix grammar: Him and me went to the park for play football.",
    "fix grammar: The informations was not accurate and needs to be corrected.",
]

print("=" * 70)
print("BASE MODEL OUTPUTS (before LoRA fine-tuning)")
print("=" * 70)
for sent in test_sentences:
    output = generate(base_model, sent)
    print(f"\nInput:  {sent}")
    print(f"Output: {output}")

BASE MODEL OUTPUTS (before LoRA fine-tuning)

Input:  fix grammar: She go to the store yesterday and buyed milk.
Output: grammatical fix grammar: She go to the store yesterday and bought milk.

Input:  fix grammar: Their is many reason why peoples should exercize daily.
Output: grammatical fix grammar: They are many reasons why peoples should exercize daily.

Input:  fix grammar: Him and me went to the park for play football.
Output: Him and me went to the park for play football.

Input:  fix grammar: The informations was not accurate and needs to be corrected.
Output: grammatical fix grammar: Die Informationen waren nicht korrekt und müssen korrigiert werden.


### Apply LoRA Adapters

Now we will apply LoRA to the model. This is the core step: they configure **which layers** get adapters and **what rank** to use, then wrap the model with PEFT.

### Key configuration parameters

| Parameter | Meaning | Value chosen |
|-----------|---------|----------|
| `r` | Rank of the low-rank matrices A and B. Lower = fewer parameters, higher = more capacity. | 16 |
| `lora_alpha` | Scaling factor for the LoRA update. Controls the magnitude of the adapter's contribution. Typically set to `r` or `2×r`. | 32 |
| `lora_dropout` | Dropout applied to LoRA layers for regularisation. Helps prevent overfitting on small datasets. | 0.05 |
| `target_modules` | Which weight matrices in the model get LoRA adapters. For T5, the query (`q`) and value (`v`) projections in attention are standard choices. | `["q", "v"]` |
| `task_type` | Tells PEFT the model architecture so it handles the forward pass correctly. | `SEQ_2_SEQ_LM` |

After applying LoRA, all original parameters are **frozen** (no gradients) and only the LoRA matrices are trainable. We will verify this by comparing parameter counts.

In [4]:
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q", "v"],
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(base_model, lora_config)

model.print_trainable_parameters()

trainable params: 589,824 || all params: 61,096,448 || trainable%: 0.9654


### Understanding the output

We should observe that only a **small fraction** (typically under 1%) of the total parameters are trainable. The rest are frozen. This is the core advantage of LoRA:

- **Less memory**: optimizer states (e.g. Adam's momentum and variance) are only needed for the trainable parameters.
- **Faster training**: fewer gradients to compute and update.
- **Less overfitting risk**: the model can't rewrite its entire knowledge base; it only learns a small task-specific adjustment.

### Inspect LoRA Layers

To make the LoRA mechanism concrete, the cell below prints the names and shapes of all **trainable** parameters. We will see that each targeted attention layer now has two small matrices: `lora_A` and `lora_B`, with the inner dimension equal to the rank `r=16`.

This is exactly the **ΔW = B · A** decomposition described earlier.

In [5]:
print(f"{'Parameter Name':<75} {'Shape':<20} {'# Params':>10}")
print("-" * 110)

total_lora_params = 0
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name:<75} {str(list(param.shape)):<20} {param.numel():>10,}")
        total_lora_params += param.numel()

print("-" * 110)
print(f"{'Total trainable (LoRA) parameters':<75} {'':<20} {total_lora_params:>10,}")

Parameter Name                                                              Shape                  # Params
--------------------------------------------------------------------------------------------------------------
base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_A.default.weight [16, 512]                 8,192
base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_B.default.weight [512, 16]                 8,192
base_model.model.encoder.block.0.layer.0.SelfAttention.v.lora_A.default.weight [16, 512]                 8,192
base_model.model.encoder.block.0.layer.0.SelfAttention.v.lora_B.default.weight [512, 16]                 8,192
base_model.model.encoder.block.1.layer.0.SelfAttention.q.lora_A.default.weight [16, 512]                 8,192
base_model.model.encoder.block.1.layer.0.SelfAttention.q.lora_B.default.weight [512, 16]                 8,192
base_model.model.encoder.block.1.layer.0.SelfAttention.v.lora_A.default.weight [16, 512]                 8,192
base

### Prepare the Training Dataset

For this demo, we will use a **small, synthetic dataset** of grammatical error correction examples. Each example has:

- **Input:** A sentence with grammatical errors, prefixed with `"fix grammar: "`.
- **Target:** The corrected sentence.

In production, one would use a larger dataset (e.g. thousands of examples from a corpus like C4-200M, JFLEG, or a domain-specific collection). Here, a small dataset is used intentionally so that training completes quickly on Colab and we can focus on the **mechanics** of the LoRA pipeline rather than waiting for long training runs.

The dataset is wrapped in a PyTorch `Dataset` class that handles tokenization.

In [ ]:
from torch.utils.data import Dataset, DataLoader

TRAIN_DATA = [
    ("fix grammar: She go to the store yesterday and buyed milk.",      # Tense
     "She went to the store yesterday and bought milk."),
    ("fix grammar: Their is many reason why peoples should exercize daily.",    # Spelling
     "There are many reasons why people should exercise daily."),
    ("fix grammar: Him and me went to the park for play football.",
     "He and I went to the park to play football."),
    ("fix grammar: The informations was not accurate and needs to be corrected.",
     "The information was not accurate and needs to be corrected."),
    ("fix grammar: I has been working here since five years.",
     "I have been working here for five years."),
    ("fix grammar: She don't knows what to do about the situation.",
     "She doesn't know what to do about the situation."),
    ("fix grammar: The childrens was playing in the garden when it started raining.",
     "The children were playing in the garden when it started raining."),
    ("fix grammar: He goed to school everyday but don't like maths.",
     "He goes to school every day but doesn't like maths."),
    ("fix grammar: Me and my friend goes to the library for studying.",
     "My friend and I go to the library to study."),
    ("fix grammar: This is one of the best book I have ever readed.",
     "This is one of the best books I have ever read."),
    ("fix grammar: The team have won there last five matches easily.",
     "The team has won their last five matches easily."),
    ("fix grammar: Each students must brings their own laptop to class.",
     "Each student must bring their own laptop to class."),
    ("fix grammar: We was suppose to finish the project by last friday.",
     "We were supposed to finish the project by last Friday."),
    ("fix grammar: Alot of people thinks that exercize is'nt important.",
     "A lot of people think that exercise isn't important."),
    ("fix grammar: The datas shows that sales has increased significant.",
     "The data shows that sales have increased significantly."),
    ("fix grammar: Neither the teacher nor the students was prepared for the test.",
     "Neither the teacher nor the students were prepared for the test."),
    ("fix grammar: He asked me that where do I live at.",
     "He asked me where I live."),
    ("fix grammar: She is more taller then her elder sister.",
     "She is taller than her elder sister."),
    ("fix grammar: I am agree with you on this matter completely.",
     "I agree with you on this matter completely."),
    ("fix grammar: The meeting will held on next monday at 10am.",
     "The meeting will be held on next Monday at 10 a.m."),
]


class GrammarDataset(Dataset):
    def __init__(self, data, tokenizer, max_input_len=128, max_target_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src, tgt = self.data[idx]

        source = self.tokenizer(
            src, max_length=self.max_input_len,
            padding="max_length", truncation=True, return_tensors="pt"
        )
        target = self.tokenizer(
            tgt, max_length=self.max_target_len,
            padding="max_length", truncation=True, return_tensors="pt"
        )

        labels = target["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": source["input_ids"].squeeze(),
            "attention_mask": source["attention_mask"].squeeze(),
            "labels": labels,
        }


train_dataset = GrammarDataset(TRAIN_DATA, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

print(f"Training examples: {len(train_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

Training examples: 20
Batches per epoch: 5


### Training Loop

The training loop below is intentionally written **from scratch** (no Hugging Face `Trainer`) so that we can see every step clearly:

1. **Forward pass**: the model receives input tokens and target labels; it computes the cross-entropy loss on the target (response) tokens.
2. **Backward pass**: gradients are computed, but **only for the LoRA parameters** (everything else is frozen, so `requires_grad=False`).
3. **Optimizer step**: the AdamW optimizer updates only the LoRA matrices A and B.
4. **Logging**: the loss is printed every few steps so that we can monitor convergence.

### Training details

- **Epochs:** 30; because the dataset is very small (20 examples), more epochs are needed for the adapter to learn the pattern. With a real dataset (thousands of examples), 1–3 epochs would suffice.
- **Learning rate:** `3e-4`; a common default for LoRA adapters; higher than typical full fine-tuning rates because only a small number of parameters are updated.
- **Optimizer:** AdamW; standard choice for transformer fine-tuning.

Expect the loss to **decrease steadily** over the epochs.

In [7]:
from torch.optim import AdamW

EPOCHS = 30
LR = 3e-4

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

model.train()

print("Starting LoRA training...")
print("=" * 50)

for epoch in range(1, EPOCHS + 1):
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3}/{EPOCHS}  |  Avg Loss: {avg_loss:.4f}")

print("=" * 50)
print("Training complete!")

Starting LoRA training...
Epoch   1/30  |  Avg Loss: 1.3483
Epoch   5/30  |  Avg Loss: 0.9529
Epoch  10/30  |  Avg Loss: 0.5730
Epoch  15/30  |  Avg Loss: 0.3272
Epoch  20/30  |  Avg Loss: 0.2783
Epoch  25/30  |  Avg Loss: 0.1842
Epoch  30/30  |  Avg Loss: 0.1578
Training complete!


### Inference: Base Model vs LoRA-Tuned Model

This is the key comparison. We will run the **same test sentences** through both the original base model and the LoRA-adapted model, side by side.

Because LoRA keeps the base weights frozen, the comparison is straightforward:

- The **base model** uses only the original pre-trained weights (LoRA contribution is disabled).
- The **LoRA model** uses the original weights **plus** the trained LoRA adapters (B·A).

Observe that the LoRA-tuned model produces **noticeably better grammatical corrections**, even though only ~0.5% of parameters were trained on just 20 examples.

In [11]:
model.eval()

test_sentences = [
    "fix grammar: She go to the store yesterday and buyed milk.",
    "fix grammar: Their is many reason why peoples should exercize daily.",
    "fix grammar: Him and me went to the park for play football.",
    "fix grammar: The informations was not accurate and needs to be corrected.",
]

unseen_sentences = [
    "fix grammar: He don't have no idea what happend at the party last nite.",
    "fix grammar: The companys profit have growed by twenty percents this year.",
]

all_test = test_sentences + unseen_sentences

print("=" * 80)
print("COMPARISON: Base Model vs LoRA-Tuned Model")
print("=" * 80)

for i, sent in enumerate(all_test):
    tag = "[TRAIN]" if i < len(test_sentences) else "[UNSEEN]"

    model.disable_adapter_layers()
    base_output = generate(model, sent)

    model.enable_adapter_layers()
    lora_output = generate(model, sent)

    print(f"\n{tag}")
    print(f"  Input:      {sent}")
    print(f"  Base model: {base_output}")
    print(f"  LoRA model: {lora_output}")

COMPARISON: Base Model vs LoRA-Tuned Model

[TRAIN]
  Input:      fix grammar: She go to the store yesterday and buyed milk.
  Base model: grammatical fix grammar: She go to the store yesterday and bought milk.
  LoRA model: She went to the store yesterday and bought milk.

[TRAIN]
  Input:      fix grammar: Their is many reason why peoples should exercize daily.
  Base model: grammatical fix grammar: They are many reasons why peoples should exercize daily.
  LoRA model: They are many reasons why people should exercise daily.

[TRAIN]
  Input:      fix grammar: Him and me went to the park for play football.
  Base model: Him and me went to the park for play football.
  LoRA model: He and I went to the park to play football.

[TRAIN]
  Input:      fix grammar: The informations was not accurate and needs to be corrected.
  Base model: grammatical fix grammar: Die Informationen waren nicht korrekt und müssen korrigiert werden.
  LoRA model: The information was not accurate and needs to be

### What to look for

We should pay attention to the following:

- **`[TRAIN]` examples:** These were in the training set. The LoRA model should handle them well, since it was trained on them directly.
- **`[UNSEEN]` examples:** These were **not** in the training set. If the LoRA model corrects them well, it has **generalised**; it learned the task pattern, not just memorised the data.
- **Base model outputs:** These show how T5-Small handles the task with zero fine-tuning. The contrast highlights exactly what LoRA adaptation added.

### Save and Load LoRA Adapters

One of the biggest practical advantages of LoRA is that the **adapter weights are tiny**, typically a few megabytes, compared to the full model (hundreds of megabytes to gigabytes). This means:

- We can **save just the adapter** and share it without distributing the full model.
- Multiple task-specific adapters can be stored alongside **one shared base model**.
- Switching tasks at inference time is as simple as loading a different adapter file.

The cell below saves the LoRA adapter weights, then demonstrates reloading them onto a fresh base model.

In [12]:
import os

ADAPTER_DIR = "lora_grammar_adapter"
model.save_pretrained(ADAPTER_DIR)

adapter_size_mb = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
) / (1024 * 1024)

print(f"Adapter saved to: {ADAPTER_DIR}/")
print(f"Adapter size: {adapter_size_mb:.2f} MB")
print(f"\nFiles saved:")
for f in os.listdir(ADAPTER_DIR):
    fpath = os.path.join(ADAPTER_DIR, f)
    if os.path.isfile(fpath):
        print(f"  {f} ({os.path.getsize(fpath) / 1024:.1f} KB)")

Adapter saved to: lora_grammar_adapter/
Adapter size: 2.27 MB

Files saved:
  adapter_model.safetensors (2313.8 KB)
  adapter_config.json (0.9 KB)
  README.md (5.0 KB)


In [13]:
from peft import PeftModel

fresh_base = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
reloaded_model = PeftModel.from_pretrained(fresh_base, ADAPTER_DIR).to(device)
reloaded_model.eval()

test_input = "fix grammar: He don't have no idea what happend at the party last nite."

print("Reloaded LoRA model output:")
print(f"  Input:  {test_input}")
print(f"  Output: {generate(reloaded_model, test_input)}")
print("\nAdapter reloaded successfully!")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Reloaded LoRA model output:
  Input:  fix grammar: He don't have no idea what happend at the party last nite.
  Output: He doesn't have any idea what happened at the party last nite.

Adapter reloaded successfully!


### Merge Adapter into Base Model

In some deployment scenarios, we may want to **merge** the LoRA weights into the base model so that inference runs at exactly the same speed as the original model, with no adapter overhead.

After merging, the model behaves as if it was fully fine-tuned, but the process that got it there was far cheaper. The merged model can then be saved as a standard Hugging Face model and deployed normally.

In [14]:
merged_model = reloaded_model.merge_and_unload()

print("LoRA adapters merged into base weights.")
print(f"Merged model parameters: {sum(p.numel() for p in merged_model.parameters()):,}")

test_input = "fix grammar: The companys profit have growed by twenty percents this year."
print(f"\nMerged model output:")
print(f"  Input:  {test_input}")
print(f"  Output: {generate(merged_model, test_input)}")

LoRA adapters merged into base weights.
Merged model parameters: 60,506,624

Merged model output:
  Input:  fix grammar: The companys profit have growed by twenty percents this year.
  Output: The company profits have grown by twenty percents this year.


### Summary 
In this notebook, we have completed a full LoRA fine-tuning pipeline:

| Step | What was done |
|------|---------------|
| **Load base model** | Loaded T5-Small (60M params) and observed its baseline performance on grammar correction. |
| **Apply LoRA** | Configured rank-16 LoRA on `q` and `v` projections; only ~0.5% of parameters became trainable. |
| **Inspect adapters** | Verified that `lora_A` and `lora_B` matrices were added with the expected shapes. |
| **Prepare data** | Created a small grammar correction dataset with tokenization and label masking. |
| **Train** | Ran a manual training loop for 30 epochs; loss decreased steadily. |
| **Compare** | Compared base vs LoRA outputs on training and unseen examples — LoRA showed clear improvement. |
| **Save / Load** | Saved the tiny adapter (~1 MB) and reloaded it onto a fresh base model. |
| **Merge** | Merged LoRA weights into the base model for zero-overhead inference. |

### Key takeaways

1. **LoRA is practical:** Fine-tuning a model with <1% trainable parameters is fast, cheap, and effective.
2. **The base model is untouched:** All general knowledge is preserved; only a small task-specific adjustment is learned.
3. **Adapters are portable:** A few megabytes of adapter weights can be saved, shared, and swapped independently of the base model.
4. **Merge is optional:** Adapters can be kept separate (for multi-task swapping) or merged (for simpler deployment).
5. **This scales:** The same workflow applies to 7B, 13B, or 70B models; LoRA makes fine-tuning accessible on limited hardware.